In [1]:
model_path = "../docs/model_phasenet_das/PhaseNet-DAS-v1.pth"

In [2]:
import torch
# append parent directory to sys.path to allow imports
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [3]:
from eqnet.models.phasenet_das import build_model

# Load the checkpoint
checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)

# Initialize the model using build_model
model = build_model(in_channels=1, out_channels=3)

# Load the state dict from checkpoint
if isinstance(checkpoint, dict) and "model" in checkpoint:
    model.load_state_dict(checkpoint["model"])
else:
    model.load_state_dict(checkpoint)

model.eval()

PhaseNetDAS(
  (backbone): UNet(
    (input_conv): Sequential(
      (enc1_conv1): Conv2d(1, 16, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3), bias=False)
      (enc1_norm1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (enc1_relu1): ReLU(inplace=True)
      (enc1_conv2): Conv2d(16, 16, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), bias=False)
      (enc1_norm2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (enc1_relu2): ReLU(inplace=True)
    )
    (encoder12): Sequential(
      (enc2_conv1): Conv2d(16, 32, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3), bias=False)
      (enc2_norm1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (enc2_relu1): ReLU(inplace=True)
      (enc2_conv2): Conv2d(32, 32, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), bias=False)
      (enc2_norm2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=

In [4]:
import torch.nn as nn

# Create a wrapper that takes tensor input instead of dict
class PhaseNetDASWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    
    def forward(self, data):
        # Wrap the input tensor in a dict as expected by PhaseNetDAS
        output = self.model({"data": data})
        # Return only the phase predictions
        return output["phase"]

# Wrap the model
wrapped_model = PhaseNetDASWrapper(model)
wrapped_model.eval()

# Create dummy input with proper dimensions
# The model uses init_stride=(4,4) and stride=(4,4), so dimensions should be divisible by powers of 2
# Shape: (batch_size, channels, time, space)
# Using dimensions that work well with the UNet architecture
dummy_data = torch.randn(1, 1, 1024, 5120)  # 3072=3*1024, 5120=5*1024

# Test the model first to make sure it runs
print("Testing model forward pass...")
with torch.no_grad():
    test_output = wrapped_model(dummy_data)
print(f"Model output shape: {test_output.shape}")

# Export to ONNX using the legacy exporter (dynamo=False)
print("Exporting to ONNX...")
torch.onnx.export(
    wrapped_model,
    dummy_data,
    "phasenet_das_test.onnx",
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["data"],
    output_names=["phase"],
    dynamic_axes={
        "data": {0: "batch_size", 2: "time", 3: "space"},
        "phase": {0: "batch_size", 2: "time", 3: "space"}
    },
    dynamo=False  # Use legacy exporter
)

# print("Model exported to phasenet_das.onnx")

Testing model forward pass...
Model output shape: torch.Size([1, 3, 1024, 5120])
Exporting to ONNX...


/tmp/ipykernel_138079/2331506313.py:33: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(
/home/aung/workspace/EQNet/eqnet/models/unet.py:25: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  pad_mode = "reflect" if padding < nt else